# AdaFace × QMUL-SurvFace — Avaliação Completa (Grupos A–I)
## Super-Resolução · Enhancement · TTA · Ensemble · SCRFD

Notebook unificado combinando **todos os experimentos** das análises v2 e v3.

| Grupo | Estratégia | Métodos |
|-------|-----------|---------|
| **A** | SR Puro | Bicubic, Lanczos, ESPCN, BSRGAN, Real-ESRGAN, SwinIR |
| **B** | SR + Upscale 2× | SR 4× seguido de resize bicúbico 2× adicional |
| **C** | SR + Upscale 4× | SR 4× seguido de resize bicúbico 4× adicional |
| **D** | Face-Specific SR | GFPGAN v1.4, CodeFormer, Real-ESRGAN→GFPGAN |
| **E** | Image Enhancement | CLAHE, Unsharp Mask, Denoising — sem SR |
| **F** | Test-Time Augmentation | Múltiplos embeddings por imagem → média |
| **G** | Ensemble de Embeddings | Preprocessings complementares → embeddings médios |
| **H** | Detector SCRFD | Substitui MTCNN por SCRFD — melhor alinhamento |
| **I** | Pipelines Combinadas | Melhores candidatos de E+F+G+H juntos |

> **Baseline de referência:** AUC=**0.6992** | TAR@FAR=0.1: **0.2878** | TAR@FAR=0.01: **0.0455**


## 1. Instalação de Dependências

In [ ]:
import os, sys, subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "opencv-contrib-python-headless"], check=False)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "facenet-pytorch", "scipy", "scikit-learn", "matplotlib",
    "tqdm", "pandas", "basicsr", "facexlib", "gfpgan",
    "realesrgan", "timm",
    "insightface", "onnxruntime-gpu"], check=False)

print("Dependências instaladas!")


## 2. Clonar Repositórios e Baixar Pesos

In [ ]:
import os, sys

# ── AdaFace (net.py necessário) ───────────────────────────────────────────
if not os.path.exists("AdaFace"):
    os.system("git clone https://github.com/mk-minchul/AdaFace.git")
sys.path.insert(0, "AdaFace")

os.makedirs("weights", exist_ok=True)

# ── ESPCN ─────────────────────────────────────────────────────────────────
if not os.path.exists("weights/ESPCN_x4.pb"):
    os.system("wget -q https://github.com/fannymonori/TF-ESPCN/raw/master/export/ESPCN_x4.pb -P weights/")
    print("ESPCN baixado")

# ── BSRGAN ────────────────────────────────────────────────────────────────
if not os.path.exists("weights/BSRGAN.pth"):
    os.system("wget -q https://github.com/cszn/BSRGAN/releases/download/v1.0/BSRGAN.pth -P weights/")
    print("BSRGAN baixado")

# ── Real-ESRGAN ───────────────────────────────────────────────────────────
if not os.path.exists("weights/RealESRGAN_x4plus.pth"):
    os.system("wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P weights/")
    print("Real-ESRGAN baixado")

# ── SwinIR ────────────────────────────────────────────────────────────────
if not os.path.exists("weights/003_realSR_BSRGAN_DFOWMFC_s64w8_SwinIR-L_x4_GAN.pth"):
    os.system("wget -q https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/003_realSR_BSRGAN_DFOWMFC_s64w8_SwinIR-L_x4_GAN.pth -P weights/")
    print("SwinIR baixado")

if not os.path.exists("SwinIR"):
    os.system("git clone https://github.com/JingyunLiang/SwinIR.git")
if "SwinIR" not in sys.path:
    sys.path.insert(0, "SwinIR")

# ── GFPGAN v1.4 ───────────────────────────────────────────────────────────
if not os.path.exists("weights/GFPGANv1.4.pth"):
    os.system("wget -q https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth -P weights/")
    print("GFPGAN v1.4 baixado")

print("\nPesos prontos!")


## 3. CodeFormer (Opcional — Grupo D)

O CodeFormer requer seu repositório próprio. Caso falhe na inicialização,
os experimentos do grupo D que o utilizam são ignorados graciosamente.


In [ ]:
import os, sys, traceback

CODEFORMER_OK = False

try:
    if not os.path.exists("CodeFormer"):
        os.system("git clone https://github.com/sczhou/CodeFormer.git")

    # Instalar dependências
    if os.path.exists("CodeFormer/requirements.txt"):
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "-r", "CodeFormer/requirements.txt"], check=False)

    if "CodeFormer" not in sys.path:
        sys.path.insert(0, "CodeFormer")

    # Baixar pesos
    w_path = "weights/codeformer.pth"
    if not os.path.exists(w_path):
        os.system("wget -q https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer.pth -P weights/")

    CODEFORMER_OK = True
    print("CodeFormer configurado!")
except Exception as e:
    print(f"CodeFormer FALHOU (será ignorado nos experimentos): {e}")

print(f"CODEFORMER_OK={CODEFORMER_OK}")


## 4. Patch de Compatibilidade (basicsr + PyTorch ≥ 2.0)

In [ ]:
import subprocess, glob

# basicsr usa torchvision.transforms.functional_tensor (removido no PyTorch 2+)
paths_to_patch = (
    glob.glob('/home/**/.venv/lib/python*/site-packages/basicsr/data/degradations.py', recursive=True) +
    glob.glob('/home/**/Documentos/**/.venv/lib/python*/site-packages/basicsr/data/degradations.py', recursive=True) +
    glob.glob('/usr/local/lib/python*/dist-packages/basicsr/data/degradations.py') +
    glob.glob('/usr/lib/python*/dist-packages/basicsr/data/degradations.py')
)

for p in set(paths_to_patch):
    subprocess.run([
        "sed", "-i",
        "s/from torchvision.transforms.functional_tensor import rgb_to_grayscale/"
        "from torchvision.transforms.functional import rgb_to_grayscale/",
        p
    ], check=False)
    print(f"Patch aplicado: {p}")

if not paths_to_patch:
    print("degradations.py não encontrado (pode já estar correto).")


## 5. Imports e Configuração

In [ ]:
import torch, cv2, numpy as np, scipy.io as sio
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_curve, auc as sk_auc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


## 6. Caminhos do Dataset

In [ ]:
DATASET_ROOT = Path("Datasets/QMUL-SurvFace")
VERIF_DIR    = DATASET_ROOT / "Face_Verification_Test_Set"
VERIF_IMGS   = VERIF_DIR / "verification_images"
POS_MAT      = VERIF_DIR / "positive_pairs_names.mat"
NEG_MAT      = VERIF_DIR / "negative_pairs_names.mat"
MODEL_PATH   = Path("adaface_ir101_webface12m.ckpt")

for p in [VERIF_IMGS, POS_MAT, NEG_MAT, MODEL_PATH]:
    status = "✓ OK" if p.exists() else "✗ NÃO ENCONTRADO"
    print(f"{status}  {p}")


## 7. Carregar Modelo AdaFace IR-101

In [ ]:
from net import build_model

def load_adaface(path, arch="ir_101"):
    m = build_model(arch)
    sd = torch.load(path, map_location=device)["state_dict"]
    msd = {k[6:]: v for k, v in sd.items() if k.startswith("model.")}
    m.load_state_dict(msd)
    return m.to(device).eval()

adaface = load_adaface(MODEL_PATH)
print("AdaFace IR-101/WebFace12M carregado!")


## 8. Carregar Detectores (MTCNN + SCRFD)

In [ ]:
import traceback

# ── MTCNN ─────────────────────────────────────────────────────────────────
from facenet_pytorch import MTCNN
mtcnn = MTCNN(image_size=112, margin=0, post_process=False, device=device)
print("MTCNN pronto.")

# ── SCRFD (buffalo_l) ─────────────────────────────────────────────────────
SCRFD_OK  = False
SCRFD_APP = None

try:
    from insightface.app import FaceAnalysis
    from insightface.utils import face_align as insightface_align

    SCRFD_APP = FaceAnalysis(
        name="buffalo_l",
        allowed_modules=["detection"],
        providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
    )
    SCRFD_APP.prepare(ctx_id=0, det_size=(640, 640))
    SCRFD_OK = True
    print("SCRFD-10G (buffalo_l) pronto.")
except BaseException as e:
    print(f"SCRFD FALHOU: {e}")

print(f"\nMTCNN=OK | SCRFD={SCRFD_OK}")


## 9. Carregar Modelos de Super-Resolução

In [ ]:
import traceback

ESPCN_OK      = False
BSRGAN_OK     = False
REALESRGAN_OK = False
SWINIR_OK     = False
GFPGAN_OK     = False

# ── ESPCN (cv2 DNN SuperRes) ─────────────────────────────────────────────
try:
    from cv2 import dnn_superres
    espcn_sr = dnn_superres.DnnSuperResImpl_create()
    espcn_sr.readModel("weights/ESPCN_x4.pb")
    espcn_sr.setModel("espcn", 4)
    ESPCN_OK = True
    print("ESPCN x4 pronto.")
except BaseException as e:
    print(f"ESPCN FALHOU: {e}")

# ── BSRGAN (basicsr RRDB) ─────────────────────────────────────────────────
try:
    from basicsr.archs.rrdbnet_arch import RRDBNet

    bsrgan_net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                          num_block=23, num_grow_ch=32, scale=4)
    raw = torch.load("weights/BSRGAN.pth", map_location=device)

    # O checkpoint BSRGAN usa formato KAIR (chaves diferentes do basicsr)
    if isinstance(raw, dict) and "params_ema" in raw:
        sd = raw["params_ema"]
    elif isinstance(raw, dict) and "params" in raw:
        sd = raw["params"]
    else:
        # Formato KAIR direto: conv_first.weight, etc.
        sd = raw if isinstance(raw, dict) else raw.state_dict()

    # Remap KAIR → basicsr: "conv_first" → "conv_first", mas body.0 → body.0
    # Tentar carregar direto primeiro; se falhar, remapear
    try:
        bsrgan_net.load_state_dict(sd, strict=True)
    except RuntimeError:
        # Remap: KAIR usa prefixos numéricos sequenciais no body
        new_sd = {}
        for k, v in sd.items():
            nk = k.replace("model.1.", "body.").replace("model.", "")
            new_sd[nk] = v
        bsrgan_net.load_state_dict(new_sd, strict=False)

    bsrgan_net = bsrgan_net.to(device).eval()
    BSRGAN_OK = True
    print("BSRGAN pronto.")
except BaseException as e:
    print(f"BSRGAN FALHOU: {e}")
    traceback.print_exc()

# ── Real-ESRGAN ───────────────────────────────────────────────────────────
try:
    from realesrgan import RealESRGANer
    from basicsr.archs.rrdbnet_arch import RRDBNet as _RRDB

    _rnet = _RRDB(num_in_ch=3, num_out_ch=3, num_feat=64,
                   num_block=23, num_grow_ch=32, scale=4)
    realesrgan_up = RealESRGANer(
        scale=4,
        model_path="weights/RealESRGAN_x4plus.pth",
        model=_rnet,
        tile=0,
        tile_pad=10,
        pre_pad=0,
        half=torch.cuda.is_available()
    )
    REALESRGAN_OK = True
    print("Real-ESRGAN x4 pronto.")
except BaseException as e:
    print(f"Real-ESRGAN FALHOU: {e}")
    traceback.print_exc()

# ── SwinIR ────────────────────────────────────────────────────────────────
try:
    from models.network_swinir import SwinIR as _SwinIR

    swinir_net = _SwinIR(
        upscale=4, in_chans=3, img_size=64, window_size=8,
        img_range=1., depths=[6,6,6,6,6,6,6,6,6],
        embed_dim=240, num_heads=[8,8,8,8,8,8,8,8,8],
        mlp_ratio=2, upsampler="nearest+conv", resi_connection="3conv"
    )
    swinir_ckpt = torch.load(
        "weights/003_realSR_BSRGAN_DFOWMFC_s64w8_SwinIR-L_x4_GAN.pth",
        map_location=device
    )
    swinir_sd = swinir_ckpt.get("params_ema", swinir_ckpt.get("params", swinir_ckpt))
    swinir_net.load_state_dict(swinir_sd, strict=True)
    swinir_net = swinir_net.to(device).eval()
    SWINIR_OK = True
    print("SwinIR-L x4 pronto.")
except BaseException as e:
    print(f"SwinIR FALHOU: {e}")
    traceback.print_exc()

print(f"\nESPCN={ESPCN_OK} | BSRGAN={BSRGAN_OK} | RealESRGAN={REALESRGAN_OK} | SwinIR={SWINIR_OK}")


## 10. Carregar GFPGAN e CodeFormer

In [ ]:
import traceback

# ── GFPGAN v1.4 ───────────────────────────────────────────────────────────
try:
    from gfpgan import GFPGANer
    gfpgan_restorer = GFPGANer(
        model_path="weights/GFPGANv1.4.pth",
        upscale=1,
        arch="clean",
        channel_multiplier=2,
        bg_upsampler=None
    )
    GFPGAN_OK = True
    print("GFPGAN v1.4 pronto.")
except BaseException as e:
    print(f"GFPGAN FALHOU: {e}")
    traceback.print_exc()

# ── CodeFormer ────────────────────────────────────────────────────────────
CODEFORMER_NET = None

if CODEFORMER_OK:
    try:
        import sys
        if "CodeFormer" not in sys.path:
            sys.path.insert(0, "CodeFormer")
        from basicsr.utils import img2tensor, tensor2img
        from basicsr.utils.download_util import load_file_from_url
        from facelib.utils.face_restoration_helper import FaceRestoreHelper
        from basicsr.archs.rrdbnet_arch import RRDBNet
        # CodeFormer usa sua própria arquitetura
        from basicsr.utils.registry import ARCH_REGISTRY
        codeformer_arch = ARCH_REGISTRY.get("CodeFormer")

        if codeformer_arch is None:
            raise RuntimeError("Arquitetura CodeFormer não encontrada no registry.")

        CODEFORMER_NET = codeformer_arch(
            dim_embd=512, codebook_size=1024, n_head=8, n_layers=9,
            connect_list=["32", "64", "128", "256"]
        ).to(device)
        ckpt = torch.load("weights/codeformer.pth", map_location=device)
        CODEFORMER_NET.load_state_dict(ckpt["params_ema"])
        CODEFORMER_NET.eval()
        print("CodeFormer pronto.")
    except BaseException as e:
        print(f"CodeFormer NET FALHOU: {e}")
        CODEFORMER_NET = None
        traceback.print_exc()

print(f"\nGFPGAN={GFPGAN_OK} | CodeFormer={'OK' if CODEFORMER_NET is not None else 'FALHOU'}")


## 11. Carregar Pares de Verificação

In [ ]:
def load_pairs(mat_path):
    data = sio.loadmat(mat_path)
    key  = [k for k in data if not k.startswith("_")][0]
    pairs = data[key]
    result = []
    for row in pairs:
        try:
            a = str(row[0][0]) if hasattr(row[0], "__len__") else str(row[0])
            b = str(row[1][0]) if hasattr(row[1], "__len__") else str(row[1])
        except Exception:
            a, b = str(row[0]), str(row[1])
        result.append((a, b))
    return result

pos_pairs = load_pairs(POS_MAT)
neg_pairs = load_pairs(NEG_MAT)
all_imgs  = set(x for p in pos_pairs + neg_pairs for x in p)
print(f"Positivos: {len(pos_pairs)} | Negativos: {len(neg_pairs)} | Imagens únicas: {len(all_imgs)}")


## 12. Funções de Super-Resolução (Grupos A–D)

Cada função recebe uma imagem BGR e retorna a versão super-resolvida (geralmente 4× maior).
Se o modelo não estiver disponível, retorna a imagem original sem modificação.


In [ ]:
import torch.nn.functional as F_nn

def _sr_bicubic(img_bgr, scale=4):
    """Resize bicúbico simples (interpolação, sem rede neural)."""
    h, w = img_bgr.shape[:2]
    return cv2.resize(img_bgr, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)

def _sr_lanczos(img_bgr, scale=4):
    """Resize Lanczos4 (melhor preservação de bordas que bicúbico)."""
    h, w = img_bgr.shape[:2]
    return cv2.resize(img_bgr, (w * scale, h * scale), interpolation=cv2.INTER_LANCZOS4)

def _sr_espcn(img_bgr):
    """ESPCN 4× via cv2.dnn_superres."""
    if not ESPCN_OK:
        return img_bgr
    try:
        return espcn_sr.upsample(img_bgr)
    except Exception:
        return img_bgr

def _sr_bsrgan(img_bgr):
    """BSRGAN 4× via basicsr RRDBNet."""
    if not BSRGAN_OK:
        return img_bgr
    try:
        rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.
        t = torch.from_numpy(rgb).permute(2,0,1).unsqueeze(0).to(device)
        with torch.no_grad():
            out = bsrgan_net(t).squeeze(0).clamp(0, 1)
        arr = (out.permute(1,2,0).cpu().numpy() * 255).astype(np.uint8)
        return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
    except Exception:
        return img_bgr

def _sr_realesrgan(img_bgr):
    """Real-ESRGAN 4× via realesrgan wrapper."""
    if not REALESRGAN_OK:
        return img_bgr
    try:
        out, _ = realesrgan_up.enhance(img_bgr, outscale=4)
        return out
    except Exception:
        return img_bgr

def _sr_swinir(img_bgr):
    """SwinIR-L 4× com padding para múltiplos de window_size=8."""
    if not SWINIR_OK:
        return img_bgr
    try:
        rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.
        t = torch.from_numpy(rgb).permute(2,0,1).unsqueeze(0).to(device)
        _, _, h, w = t.shape
        window_size = 8
        # Pad para múltiplo de window_size
        h_pad = (h // window_size + 1) * window_size - h
        w_pad = (w // window_size + 1) * window_size - w
        t_pad = F_nn.pad(t, (0, w_pad, 0, h_pad), "reflect")
        with torch.no_grad():
            out = swinir_net(t_pad)
        # Crop padding ampliado 4×
        out = out[:, :, :h*4, :w*4]
        arr = (out.squeeze(0).clamp(0,1).permute(1,2,0).cpu().numpy() * 255).astype(np.uint8)
        return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
    except Exception:
        return img_bgr

def _apply_gfpgan(img_bgr):
    """GFPGAN v1.4 (face-specific SR, upscale=1)."""
    if not GFPGAN_OK:
        return img_bgr
    try:
        _, _, out = gfpgan_restorer.enhance(
            img_bgr, has_aligned=False, only_center_face=True, paste_back=True)
        return out if out is not None else img_bgr
    except Exception:
        return img_bgr

def _apply_codeformer(img_bgr, w=0.5):
    """CodeFormer com parâmetro de fidelidade w (0=qualidade, 1=fidelidade)."""
    if CODEFORMER_NET is None:
        return img_bgr
    try:
        from basicsr.utils import img2tensor, tensor2img
        from facelib.utils.face_restoration_helper import FaceRestoreHelper

        face_helper = FaceRestoreHelper(
            upscale_factor=1, face_size=512, crop_ratio=(1, 1),
            det_model="retinaface_resnet50", save_ext="png",
            use_parse=True, device=device
        )
        face_helper.clean_all()
        face_helper.read_image(img_bgr)
        face_helper.get_face_landmarks_5(only_center_face=True, resize=640)
        face_helper.align_warp_face()

        for i, cropped in enumerate(face_helper.cropped_faces):
            t = img2tensor(cropped / 255., bgr2rgb=True, float32=True).unsqueeze(0).to(device)
            with torch.no_grad():
                out = CODEFORMER_NET(t, w=w, adain=True)[0]
            face_helper.add_restored_face(tensor2img(out.squeeze(0), rgb2bgr=True))

        face_helper.get_inverse_affine(None)
        restored = face_helper.paste_faces_to_input_image()
        return restored if restored is not None else img_bgr
    except Exception:
        return img_bgr

print("Funções de SR prontas.")


## 13. Funções de Image Enhancement (Grupos E–I)

Trabalham no espaço **LAB** para preservar cromaticidade, modificando apenas
o canal de luminância (L). Sem upscale — a imagem permanece no mesmo tamanho.


In [ ]:
def _clahe(img_bgr, clip_limit=2.0, tile=(8, 8)):
    """CLAHE no canal L do espaço LAB. Melhora contraste local."""
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    cl = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile).apply(l)
    return cv2.cvtColor(cv2.merge([cl, a, b]), cv2.COLOR_LAB2BGR)

def _unsharp(img_bgr, sigma=0.5, amount=1.5):
    """Unsharp masking: realça bordas sem inventar texturas."""
    blurred = cv2.GaussianBlur(img_bgr, (0, 0), sigma)
    sharp = (1 + amount) * img_bgr.astype(np.float32) - amount * blurred.astype(np.float32)
    return np.clip(sharp, 0, 255).astype(np.uint8)

def _denoise(img_bgr, h=3):
    """Non-local means denoising apenas no canal L. Preserva cor e geometria."""
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l_den = cv2.fastNlMeansDenoising(l, None, h=float(h),
                                      templateWindowSize=7, searchWindowSize=21)
    return cv2.cvtColor(cv2.merge([l_den, a, b]), cv2.COLOR_LAB2BGR)

# ── Dispatcher Unificado ──────────────────────────────────────────────────
def apply_preproc(img_bgr, key):
    """
    Aplica pré-processamento identificado por 'key'.
    Retorna sempre imagem BGR (pode ser maior que a original para SR 4x).
    """
    if key is None or key == "baseline":
        return img_bgr
    # ── SR Puro (Grupo A) ────────────────────────────────────────────────
    elif key == "bicubic_4x":
        return _sr_bicubic(img_bgr, 4)
    elif key == "lanczos_4x":
        return _sr_lanczos(img_bgr, 4)
    elif key == "espcn_4x":
        return _sr_espcn(img_bgr)
    elif key == "bsrgan_4x":
        return _sr_bsrgan(img_bgr)
    elif key == "realesrgan_4x":
        return _sr_realesrgan(img_bgr)
    elif key == "swinir_4x":
        return _sr_swinir(img_bgr)
    # ── SR + Upscale Adicional (Grupos B e C) ────────────────────────────
    elif key == "espcn_4x_2x":
        return _sr_bicubic(_sr_espcn(img_bgr), 2)
    elif key == "bsrgan_4x_2x":
        return _sr_bicubic(_sr_bsrgan(img_bgr), 2)
    elif key == "realesrgan_4x_2x":
        return _sr_bicubic(_sr_realesrgan(img_bgr), 2)
    elif key == "swinir_4x_2x":
        return _sr_bicubic(_sr_swinir(img_bgr), 2)
    elif key == "espcn_4x_4x":
        return _sr_bicubic(_sr_espcn(img_bgr), 4)
    elif key == "bsrgan_4x_4x":
        return _sr_bicubic(_sr_bsrgan(img_bgr), 4)
    elif key == "realesrgan_4x_4x":
        return _sr_bicubic(_sr_realesrgan(img_bgr), 4)
    elif key == "swinir_4x_4x":
        return _sr_bicubic(_sr_swinir(img_bgr), 4)
    # ── Face-Specific (Grupo D) ──────────────────────────────────────────
    elif key == "gfpgan":
        return _apply_gfpgan(img_bgr)
    elif key == "codeformer_w05":
        return _apply_codeformer(img_bgr, w=0.5)
    elif key == "codeformer_w10":
        return _apply_codeformer(img_bgr, w=1.0)
    elif key == "realesrgan_gfpgan":
        return _apply_gfpgan(_sr_realesrgan(img_bgr))
    # ── Enhancement (Grupos E–I) ─────────────────────────────────────────
    elif key == "clahe":
        return _clahe(img_bgr, 2.0, (8, 8))
    elif key == "clahe_strong":
        return _clahe(img_bgr, 4.0, (4, 4))
    elif key == "unsharp_mild":
        return _unsharp(img_bgr, 0.5, 1.5)
    elif key == "unsharp_strong":
        return _unsharp(img_bgr, 1.0, 2.0)
    elif key == "clahe_unsharp":
        return _unsharp(_clahe(img_bgr, 2.0, (8, 8)), 0.5, 1.5)
    elif key == "denoise_light":
        return _denoise(img_bgr, 3)
    elif key == "denoise_strong":
        return _denoise(img_bgr, 7)
    elif key == "denoise_clahe":
        return _clahe(_denoise(img_bgr, 3), 2.0, (8, 8))
    # ── Alias para ensemble ──────────────────────────────────────────────
    elif key == "espcn":
        return _sr_espcn(img_bgr)
    return img_bgr

# ── Augmentações para TTA (aplicadas APÓS alignment 112×112) ─────────────
TTA_FNS = {
    "flip":      lambda x: cv2.flip(x, 1),
    "bright_up": lambda x: np.clip(x.astype(np.int16) + 15, 0, 255).astype(np.uint8),
    "bright_dn": lambda x: np.clip(x.astype(np.int16) - 15, 0, 255).astype(np.uint8),
    "clahe_aug": lambda x: _clahe(x, 2.0, (8, 8)),
}

print("Funções de pré-processamento e TTA prontas.")


## 14. Funções de Alinhamento Facial

- **MTCNN**: detector padrão. Para imagens SR (já ampliadas 4×), o MTCNN tem mais
  resolução para localizar landmarks. Para imagens originais (~20px), geralmente
  falha e usa fallback de resize bicúbico para 112×112 (Skipped=0 garantido).
  
  > **Fix crítico:** `mtcnn(pil)` lança `ValueError: torch.cat()` em imagens
  > com <~15px porque a pirâmide de escalas gera boxes vazio. Capturado com
  > `try/except` e redirecionado para o fallback.

- **SCRFD**: faz upscale 4× antes de detectar, depois alinha com os landmarks.
  Fallback em cascata: SCRFD → MTCNN → resize.


In [ ]:
def _fallback_resize(img_bgr):
    """Fallback: resize bicúbico da imagem inteira para 112×112."""
    return cv2.resize(img_bgr, (112, 112), interpolation=cv2.INTER_CUBIC)

def _align_mtcnn(img_bgr):
    """
    Detecta e alinha com MTCNN.
    Retorna sempre BGR 112×112.
    try/except captura ValueError em imagens <~15px (torch.cat lista vazia).
    """
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    try:
        face_t = mtcnn(pil)
    except Exception:
        face_t = None  # imagem muito pequena para MTCNN processar
    if face_t is None:
        return _fallback_resize(img_bgr)
    arr = face_t.permute(1, 2, 0).cpu().clamp(0, 255).numpy().astype(np.uint8)
    return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)

def _align_scrfd(img_bgr):
    """
    Detecta e alinha com SCRFD (buffalo_l).
    Upscala 4× antes de detectar para melhorar localização em faces ~20px.
    Fallback: SCRFD → MTCNN → resize.
    """
    if not SCRFD_OK or SCRFD_APP is None:
        return _align_mtcnn(img_bgr)

    h, w = img_bgr.shape[:2]
    det_img = cv2.resize(img_bgr, (w * 4, h * 4), interpolation=cv2.INTER_CUBIC)

    try:
        faces = SCRFD_APP.get(det_img)
    except Exception:
        return _align_mtcnn(img_bgr)

    if not faces:
        return _align_mtcnn(img_bgr)

    face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))

    try:
        aligned = insightface_align.norm_crop(det_img, landmark=face.kps, image_size=112)
        return aligned
    except Exception:
        return _align_mtcnn(img_bgr)

def get_aligned_face(img_bgr, detector="mtcnn"):
    if detector == "scrfd":
        return _align_scrfd(img_bgr)
    return _align_mtcnn(img_bgr)

print("Funções de alinhamento prontas (Skipped=0 garantido).")


## 15. Extração de Embedding (com TTA)

In [ ]:
def embed_face(aligned_bgr):
    """Normaliza e extrai embedding AdaFace de face 112×112 BGR."""
    rgb = cv2.cvtColor(aligned_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.
    t = torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0).to(device)
    t = (t - 0.5) / 0.5  # AdaFace espera [-1, 1]
    with torch.no_grad():
        emb, _ = adaface(t)
    emb = emb.cpu().numpy()[0]
    return emb / (np.linalg.norm(emb) + 1e-8)

def embed_with_tta(aligned_bgr, tta_keys):
    """
    Extrai embedding com Test-Time Augmentation.
    Aplica cada augmentação, coleta embeddings e retorna a média L2-normalizada.
    """
    embs = [embed_face(aligned_bgr)]
    for key in tta_keys:
        aug_fn = TTA_FNS.get(key)
        if aug_fn is None:
            continue
        try:
            aug = aug_fn(aligned_bgr)
            embs.append(embed_face(aug))
        except Exception:
            pass
    avg = np.mean(embs, axis=0)
    return avg / (np.linalg.norm(avg) + 1e-8)

print("Funções de embedding prontas.")


## 16. Cache de Faces e Executor de Experimentos

O **face_cache** evita re-detectar faces para experimentos que compartilham
o mesmo pré-processamento e detector. Chave: `(img_path, preproc_key, detector)`.


In [ ]:
import collections, traceback

face_cache = {}   # (img_path, preproc_key, detector) -> aligned BGR 112x112 | None
_error_log  = collections.Counter()

def _get_cached_face(img_path, preproc_key, detector):
    cache_key = (img_path, preproc_key or "baseline", detector)
    if cache_key not in face_cache:
        try:
            img = cv2.imread(img_path)
            if img is None:
                _error_log["imread_None"] += 1
                face_cache[cache_key] = None
            else:
                proc = apply_preproc(img, preproc_key)
                face_cache[cache_key] = get_aligned_face(proc, detector)
        except Exception as e:
            _error_log[type(e).__name__ + ": " + str(e)[:80]] += 1
            face_cache[cache_key] = None
    return face_cache[cache_key]

def run_single(img_path, preproc_key, detector, tta_keys):
    """Pipeline simples: preproc → detect/align → TTA → embedding."""
    aligned = _get_cached_face(img_path, preproc_key, detector)
    if aligned is None:
        return None
    return embed_with_tta(aligned, tta_keys)

def run_ensemble(img_path, preproc_keys, detector, tta_keys):
    """
    Ensemble: cada preproc_key gera um embedding (com TTA).
    Todos os embeddings são médios e re-normalizados.
    """
    embs = []
    for pk in preproc_keys:
        aligned = _get_cached_face(img_path, pk, detector)
        if aligned is None:
            continue
        emb = embed_with_tta(aligned, tta_keys)
        embs.append(emb)
    if not embs:
        return None
    avg = np.mean(embs, axis=0)
    return avg / (np.linalg.norm(avg) + 1e-8)

def get_embedding(img_path, exp):
    """Dispatcher central: executa o experimento 'exp' sobre img_path."""
    tta_keys = exp.get("tta") or []
    detector = exp.get("detector", "mtcnn")
    ensemble = exp.get("ensemble")
    if ensemble is not None:
        return run_ensemble(img_path, ensemble, detector, tta_keys)
    return run_single(img_path, exp.get("preproc"), detector, tta_keys)

print("Cache e executor prontos.")


## 17. Função de Métricas

In [ ]:
def compute_metrics(pos_scores, neg_scores):
    """Calcula AUC, TAR@FAR, Best Accuracy a partir de scores coseno."""
    labels = np.array([1]*len(pos_scores) + [0]*len(neg_scores))
    scores = np.array(pos_scores + neg_scores)

    fpr, tpr, thr = roc_curve(labels, scores)
    auc_val = sk_auc(fpr, tpr)

    idx01  = np.searchsorted(fpr, 0.1,  side="right") - 1
    tar01  = tpr[idx01]  if idx01  >= 0 else 0.
    idx001 = np.searchsorted(fpr, 0.01, side="right") - 1
    tar001 = tpr[idx001] if idx001 >= 0 else 0.

    accs = tpr * len(pos_scores) + (1 - fpr) * len(neg_scores)
    accs /= (len(pos_scores) + len(neg_scores))
    bacc = float(np.max(accs))

    return dict(auc=auc_val, tar01=tar01, tar001=tar001, bacc=bacc, fpr=fpr, tpr=tpr)

print("compute_metrics pronta.")


## 18. Configuração de Todos os Experimentos (Grupos A–I)

| Campo | Descrição |
|-------|-----------|
| `name` | Rótulo no gráfico/tabela |
| `group` | Grupo (A, B, C, D, E, F, G, H, I) |
| `preproc` | Chave de pré-processamento (`None` = baseline) |
| `ensemble` | Lista de chaves para ensemble (ou `None`) |
| `detector` | `"mtcnn"` ou `"scrfd"` |
| `tta` | Lista de augmentações TTA após alignment (ou `None`) |


In [ ]:
EXPERIMENTS = [
    # ── Referência ──────────────────────────────────────────────────────────
    dict(name="Baseline",              group="Ref", preproc=None,              ensemble=None,                       detector="mtcnn", tta=None),

    # ── Grupo A: Super-Resolução Pura ───────────────────────────────────────
    dict(name="Bicubic 4x",            group="A",   preproc="bicubic_4x",      ensemble=None,                       detector="mtcnn", tta=None),
    dict(name="Lanczos 4x",            group="A",   preproc="lanczos_4x",      ensemble=None,                       detector="mtcnn", tta=None),
    dict(name="ESPCN 4x",              group="A",   preproc="espcn_4x",        ensemble=None,                       detector="mtcnn", tta=None),
    dict(name="BSRGAN 4x",             group="A",   preproc="bsrgan_4x",       ensemble=None,                       detector="mtcnn", tta=None),
    dict(name="Real-ESRGAN 4x",        group="A",   preproc="realesrgan_4x",   ensemble=None,                       detector="mtcnn", tta=None),
    dict(name="SwinIR 4x",             group="A",   preproc="swinir_4x",       ensemble=None,                       detector="mtcnn", tta=None),

    # ── Grupo B: SR + Upscale 2x Adicional ──────────────────────────────────
    dict(name="ESPCN + 2x",            group="B",   preproc="espcn_4x_2x",     ensemble=None,                       detector="mtcnn", tta=None),
    dict(name="BSRGAN + 2x",           group="B",   preproc="bsrgan_4x_2x",    ensemble=None,                       detector="mtcnn", tta=None),
    dict(name="RealESRGAN + 2x",       group="B",   preproc="realesrgan_4x_2x",ensemble=None,                       detector="mtcnn", tta=None),
    dict(name="SwinIR + 2x",           group="B",   preproc="swinir_4x_2x",    ensemble=None,                       detector="mtcnn", tta=None),

    # ── Grupo C: SR + Upscale 4x Adicional ──────────────────────────────────
    dict(name="ESPCN + 4x",            group="C",   preproc="espcn_4x_4x",     ensemble=None,                       detector="mtcnn", tta=None),
    dict(name="BSRGAN + 4x",           group="C",   preproc="bsrgan_4x_4x",    ensemble=None,                       detector="mtcnn", tta=None),
    dict(name="RealESRGAN + 4x",       group="C",   preproc="realesrgan_4x_4x",ensemble=None,                       detector="mtcnn", tta=None),
    dict(name="SwinIR + 4x",           group="C",   preproc="swinir_4x_4x",    ensemble=None,                       detector="mtcnn", tta=None),

    # ── Grupo D: Face-Specific SR ────────────────────────────────────────────
    dict(name="GFPGAN v1.4",           group="D",   preproc="gfpgan",          ensemble=None,                       detector="mtcnn", tta=None),
    dict(name="CodeFormer w=0.5",      group="D",   preproc="codeformer_w05",  ensemble=None,                       detector="mtcnn", tta=None),
    dict(name="CodeFormer w=1.0",      group="D",   preproc="codeformer_w10",  ensemble=None,                       detector="mtcnn", tta=None),
    dict(name="RealESRGAN+GFPGAN",     group="D",   preproc="realesrgan_gfpgan",ensemble=None,                      detector="mtcnn", tta=None),

    # ── Grupo E: Image Enhancement ───────────────────────────────────────────
    dict(name="Denoise Leve",          group="E",   preproc="denoise_light",   ensemble=None,                       detector="mtcnn", tta=None),
    dict(name="Denoise Forte",         group="E",   preproc="denoise_strong",  ensemble=None,                       detector="mtcnn", tta=None),
    dict(name="Unsharp Leve",          group="E",   preproc="unsharp_mild",    ensemble=None,                       detector="mtcnn", tta=None),
    dict(name="Unsharp Forte",         group="E",   preproc="unsharp_strong",  ensemble=None,                       detector="mtcnn", tta=None),
    dict(name="CLAHE Forte",           group="E",   preproc="clahe_strong",    ensemble=None,                       detector="mtcnn", tta=None),
    dict(name="Denoise+CLAHE",         group="E",   preproc="denoise_clahe",   ensemble=None,                       detector="mtcnn", tta=None),
    dict(name="CLAHE",                 group="E",   preproc="clahe",           ensemble=None,                       detector="mtcnn", tta=None),
    dict(name="CLAHE+Unsharp",         group="E",   preproc="clahe_unsharp",   ensemble=None,                       detector="mtcnn", tta=None),

    # ── Grupo F: Test-Time Augmentation ──────────────────────────────────────
    dict(name="TTA-2 (flip)",          group="F",   preproc=None,              ensemble=None,                       detector="mtcnn", tta=["flip"]),
    dict(name="TTA-4 (flip+bright)",   group="F",   preproc=None,              ensemble=None,                       detector="mtcnn", tta=["flip","bright_up","bright_dn"]),
    dict(name="TTA-5 Full",            group="F",   preproc=None,              ensemble=None,                       detector="mtcnn", tta=["flip","bright_up","bright_dn","clahe_aug"]),
    dict(name="TTA CLAHE aug",         group="F",   preproc=None,              ensemble=None,                       detector="mtcnn", tta=["clahe_aug"]),
    dict(name="CLAHE + TTA-2",         group="F",   preproc="clahe",           ensemble=None,                       detector="mtcnn", tta=["flip"]),
    dict(name="CLAHE + TTA-5",         group="F",   preproc="clahe",           ensemble=None,                       detector="mtcnn", tta=["flip","bright_up","bright_dn","clahe_aug"]),

    # ── Grupo G: Ensemble de Embeddings ──────────────────────────────────────
    dict(name="Ens(Base+Denoise)",     group="G",   preproc=None,              ensemble=[None, "denoise_light"],     detector="mtcnn", tta=None),
    dict(name="Ens(Base+GFPGAN)",      group="G",   preproc=None,              ensemble=[None, "gfpgan"],            detector="mtcnn", tta=None),
    dict(name="Ens(Base+ESPCN)",       group="G",   preproc=None,              ensemble=[None, "espcn"],             detector="mtcnn", tta=None),
    dict(name="Ens(Base+Unsharp)",     group="G",   preproc=None,              ensemble=[None, "unsharp_mild"],      detector="mtcnn", tta=None),
    dict(name="Ens(B+CLAHE+GFPGAN)",   group="G",   preproc=None,              ensemble=[None, "clahe", "gfpgan"],   detector="mtcnn", tta=None),
    dict(name="Ens(Base+CLAHE)",       group="G",   preproc=None,              ensemble=[None, "clahe"],             detector="mtcnn", tta=None),
    dict(name="Ens(CLAHE+GFPGAN)",     group="G",   preproc=None,              ensemble=["clahe", "gfpgan"],         detector="mtcnn", tta=None),

    # ── Grupo H: SCRFD Detector ───────────────────────────────────────────────
    dict(name="SCRFD + TTA-2",         group="H",   preproc=None,              ensemble=None,                       detector="scrfd", tta=["flip"]),
    dict(name="SCRFD",                 group="H",   preproc=None,              ensemble=None,                       detector="scrfd", tta=None),
    dict(name="SCRFD + Unsharp",       group="H",   preproc="unsharp_mild",    ensemble=None,                       detector="scrfd", tta=None),
    dict(name="SCRFD + CLAHE + TTA-2", group="H",   preproc="clahe",           ensemble=None,                       detector="scrfd", tta=["flip"]),
    dict(name="SCRFD + CLAHE",         group="H",   preproc="clahe",           ensemble=None,                       detector="scrfd", tta=None),

    # ── Grupo I: Pipelines Combinadas ─────────────────────────────────────────
    dict(name="Ens(B+C+G) + TTA-2",   group="I",   preproc=None,              ensemble=[None, "clahe", "gfpgan"],   detector="mtcnn", tta=["flip"]),
    dict(name="Ens(B+C) + TTA-2",      group="I",   preproc=None,              ensemble=[None, "clahe"],             detector="mtcnn", tta=["flip"]),
    dict(name="Denoise+CLAHE + TTA-2", group="I",   preproc="denoise_clahe",   ensemble=None,                       detector="mtcnn", tta=["flip"]),
    dict(name="CLAHE + TTA-5 Full",    group="I",   preproc="clahe",           ensemble=None,                       detector="mtcnn", tta=["flip","bright_up","bright_dn","clahe_aug"]),
    dict(name="SCRFD+CLAHE + TTA-5",   group="I",   preproc="clahe",           ensemble=None,                       detector="scrfd", tta=["flip","bright_up","bright_dn","clahe_aug"]),
]

print(f"Total de experimentos: {len(EXPERIMENTS)}")
for e in EXPERIMENTS:
    det = e['detector'].upper()
    ens = f"Ens{e['ensemble']}" if e['ensemble'] else ""
    pp  = e['preproc'] or "baseline"
    tta = f"+TTA{e['tta']}" if e['tta'] else ""
    print(f"  [{e['group']}] {e['name']:35s}  {det}  {ens or pp}{tta}")


## 19. Diagnóstico de Sanidade

Testa a pipeline completa em uma imagem antes de rodar todos os experimentos.


In [ ]:
import traceback

test_img_name = sorted(all_imgs)[0]
test_path = str(VERIF_IMGS / test_img_name)
print(f"Imagem de teste: {test_path}")

img = cv2.imread(test_path)
print(f"1. cv2.imread: shape={img.shape if img is not None else 'NONE!'}")

try:
    proc = apply_preproc(img, None)
    print(f"2. apply_preproc(baseline): shape={proc.shape}")
except Exception as e:
    print(f"2. apply_preproc FALHOU: {e}")

try:
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    try:
        face_t = mtcnn(pil)
    except Exception:
        face_t = None
    print(f"3. mtcnn(): {'None (fallback)' if face_t is None else f'shape={face_t.shape} device={face_t.device}'}")
except Exception as e:
    print(f"3. MTCNN FALHOU: {e}")

try:
    aligned = _align_mtcnn(img)
    print(f"4. _align_mtcnn(): shape={aligned.shape}")
    emb = embed_face(aligned)
    print(f"5. embed_face(): shape={emb.shape}, norm={np.linalg.norm(emb):.4f}")
except Exception as e:
    print(f"4/5. FALHOU: {e}")
    traceback.print_exc()

_error_log.clear()
result = get_embedding(test_path, EXPERIMENTS[0])
print(f"6. get_embedding(Baseline): {'OK shape=' + str(result.shape) if result is not None else 'NONE!'} | errors={dict(_error_log)}")


## 20. Executar Todos os Experimentos

In [ ]:
results = {}

print(f"Imagens únicas: {len(all_imgs)}")
print(f"Experimentos: {len(EXPERIMENTS)}")
print("=" * 75)

for exp in EXPERIMENTS:
    name = exp["name"]
    print(f"\n[{exp['group']}] {name}")

    emb_cache = {}
    imgs_list = sorted(all_imgs)
    skipped = 0

    for img_name in tqdm(imgs_list, desc=name, leave=False):
        img_path = str(VERIF_IMGS / img_name)
        emb = get_embedding(img_path, exp)
        if emb is None:
            skipped += 1
        emb_cache[img_name] = emb

    pos_scores, neg_scores = [], []
    for a, b in pos_pairs:
        ea, eb = emb_cache.get(a), emb_cache.get(b)
        if ea is not None and eb is not None:
            pos_scores.append(float(np.dot(ea, eb)))
    for a, b in neg_pairs:
        ea, eb = emb_cache.get(a), emb_cache.get(b)
        if ea is not None and eb is not None:
            neg_scores.append(float(np.dot(ea, eb)))

    if len(pos_scores) < 10 or len(neg_scores) < 10:
        print(f"  AVISO: pares insuficientes (pos={len(pos_scores)}, neg={len(neg_scores)})")
        continue

    m = compute_metrics(pos_scores, neg_scores)
    m["skipped"] = skipped
    results[name] = m

    print(f"  AUC={m['auc']:.4f}  TAR@0.1={m['tar01']:.4f}  "
          f"TAR@0.01={m['tar001']:.4f}  BestAcc={m['bacc']:.4f}  "
          f"Skipped={skipped}")
    if _error_log:
        print(f"  ERROS SILENCIADOS: {dict(_error_log)}")
        _error_log.clear()

print("\n" + "=" * 75)
print(f"Concluído! {len(results)} experimentos processados.")


## 21. Salvar Resultados em CSV

In [ ]:
import os

rows = []
exp_map = {e["name"]: e for e in EXPERIMENTS}
GROUP_ORDER = ["Ref", "A", "B", "C", "D", "E", "F", "G", "H", "I"]

for name, m in results.items():
    exp = exp_map.get(name, {})
    rows.append({
        "Grupo":         exp.get("group", "?"),
        "Método":        name,
        "Detector":      exp.get("detector", "mtcnn").upper(),
        "AUC":           round(m["auc"],   4),
        "TAR@FAR=0.1":   round(m["tar01"], 4),
        "TAR@FAR=0.01":  round(m["tar001"],4),
        "Best Accuracy": round(m["bacc"],  4),
        "Pares Válidos": len(pos_pairs),
        "Skipped":       m["skipped"],
    })

df = pd.DataFrame(rows)
df["_order"] = df["Grupo"].map({g: i for i, g in enumerate(GROUP_ORDER)})
df = df.sort_values(["_order", "AUC"], ascending=[True, False]).drop(columns=["_order"])

os.makedirs("Resultados", exist_ok=True)
csv_path = "Resultados/adaface_qmul_results_combined.csv"
df.to_csv(csv_path, index=False)
print(f"CSV salvo: {csv_path}")

# Tabela resumo no terminal
baseline_auc = results.get("Baseline", {}).get("auc", 0.6992)
print("\n" + "=" * 92)
print(f"{'Grp':^5} {'Método':^35} {'AUC':^8} {'TAR@0.1':^9} {'TAR@0.01':^10} {'BestAcc':^9}")
print("─" * 92)
for _, row in df.iterrows():
    delta  = row["AUC"] - baseline_auc
    marker = " ▲" if delta > 0.001 else (" ▼" if delta < -0.001 else "  ")
    print(f"{row['Grupo']:^5} {row['Método']:35s} {row['AUC']:.4f}{marker}  "
          f"{row['TAR@FAR=0.1']:.4f}    {row['TAR@FAR=0.01']:.4f}     {row['Best Accuracy']:.4f}")
print("─" * 92)
print(f"▲ = acima do Baseline (AUC={baseline_auc:.4f})  ▼ = abaixo")


## 22. Visualização dos Resultados

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

GROUP_COLORS = {
    "Ref": "#e74c3c",
    "A":   "#2980b9",
    "B":   "#1abc9c",
    "C":   "#16a085",
    "D":   "#8e44ad",
    "E":   "#3498db",
    "F":   "#2ecc71",
    "G":   "#9b59b6",
    "H":   "#f39c12",
    "I":   "#e67e22",
}
GROUP_LABELS = {
    "Ref": "Baseline",
    "A":   "Grupo A: SR Puro",
    "B":   "Grupo B: SR + 2x",
    "C":   "Grupo C: SR + 4x",
    "D":   "Grupo D: Face-Specific SR",
    "E":   "Grupo E: Enhancement",
    "F":   "Grupo F: TTA",
    "G":   "Grupo G: Ensemble",
    "H":   "Grupo H: SCRFD",
    "I":   "Grupo I: Combinadas",
}

exp_map   = {e["name"]: e for e in EXPERIMENTS}
exp_order = [e["name"] for e in EXPERIMENTS if e["name"] in results]

fig = plt.figure(figsize=(28, 22))
fig.suptitle(
    "AdaFace IR-101/WebFace12M — QMUL-SurvFace Verificação (Grupos A–I)\n"
    "Super-Resolução × Enhancement × TTA × Ensemble × SCRFD",
    fontsize=15, fontweight="bold"
)

# ── 1. Curvas ROC (melhores por grupo) ───────────────────────────────────
ax_roc = fig.add_subplot(2, 2, 1)
ax_roc.set_title("Curvas ROC — melhores por grupo", fontsize=12)

best_by_group = {}
for name, m in results.items():
    grp = exp_map[name]["group"]
    if grp not in best_by_group or m["auc"] > best_by_group[grp][1]["auc"]:
        best_by_group[grp] = (name, m)

for name, m in results.items():
    grp = exp_map[name]["group"]
    ax_roc.plot(m["fpr"], m["tpr"], color=GROUP_COLORS.get(grp, "#aaa"),
                alpha=0.10, linewidth=0.7)

for grp, (name, m) in sorted(best_by_group.items(), key=lambda x: GROUP_ORDER.index(x[0]) if x[0] in GROUP_ORDER else 99):
    ax_roc.plot(m["fpr"], m["tpr"],
                color=GROUP_COLORS.get(grp, "#aaa"), linewidth=2,
                label=f"{name} (AUC={m['auc']:.4f})")

ax_roc.plot([0,1],[0,1],"k--",alpha=0.3)
ax_roc.set_xlabel("False Positive Rate"); ax_roc.set_ylabel("True Positive Rate")
ax_roc.legend(fontsize=7, loc="lower right")
ax_roc.set_xlim(0,1); ax_roc.set_ylim(0,1)
ax_roc.grid(True, alpha=0.3)

# ── 2. AUC por método (barras por grupo) ─────────────────────────────────
ax_bar = fig.add_subplot(2, 2, 2)
ax_bar.set_title("AUC por Pipeline", fontsize=12)

aucs   = [results[n]["auc"] for n in exp_order]
colors = [GROUP_COLORS.get(exp_map[n]["group"], "#aaa") for n in exp_order]
baseline_auc = results.get("Baseline", {}).get("auc", 0.6992)

bars = ax_bar.barh(exp_order, aucs, color=colors, edgecolor="white", height=0.75)
ax_bar.axvline(baseline_auc, color="#e74c3c", linestyle="--", linewidth=1.5, alpha=0.9)
ax_bar.set_xlabel("AUC")
ax_bar.set_xlim(max(0, min(aucs)*0.97), max(aucs)*1.005)
ax_bar.tick_params(axis='y', labelsize=6)
for bar, val in zip(bars, aucs):
    ax_bar.text(val + 0.0003, bar.get_y() + bar.get_height()/2,
                f"{val:.4f}", va="center", fontsize=5.5)

patches = [mpatches.Patch(color=c, label=GROUP_LABELS[g])
           for g, c in GROUP_COLORS.items() if g in set(exp_map[n]["group"] for n in results)]
ax_bar.legend(handles=patches, fontsize=6.5, loc="lower right")

# ── 3. TAR@FAR=0.1 ────────────────────────────────────────────────────────
ax_t1 = fig.add_subplot(2, 2, 3)
ax_t1.set_title("TAR @ FAR=0.1 por Pipeline", fontsize=12)
tar01s = [results[n]["tar01"] for n in exp_order]
bars3 = ax_t1.barh(exp_order, tar01s, color=colors, edgecolor="white", height=0.75)
ax_t1.axvline(results.get("Baseline",{}).get("tar01",0.2878), color="#e74c3c",
              linestyle="--", linewidth=1.5, alpha=0.9)
ax_t1.set_xlabel("TAR @ FAR=0.1")
ax_t1.tick_params(axis='y', labelsize=6)
for bar, val in zip(bars3, tar01s):
    ax_t1.text(val + 0.001, bar.get_y() + bar.get_height()/2,
               f"{val:.4f}", va="center", fontsize=5.5)

# ── 4. TAR@FAR=0.01 ───────────────────────────────────────────────────────
ax_t01 = fig.add_subplot(2, 2, 4)
ax_t01.set_title("TAR @ FAR=0.01 por Pipeline (threshold conservador)", fontsize=12)
tar001s = [results[n]["tar001"] for n in exp_order]
bars4 = ax_t01.barh(exp_order, tar001s, color=colors, edgecolor="white", height=0.75)
ax_t01.axvline(results.get("Baseline",{}).get("tar001",0.0455), color="#e74c3c",
               linestyle="--", linewidth=1.5, alpha=0.9)
ax_t01.set_xlabel("TAR @ FAR=0.01")
ax_t01.tick_params(axis='y', labelsize=6)
for bar, val in zip(bars4, tar001s):
    ax_t01.text(val + 0.0003, bar.get_y() + bar.get_height()/2,
                f"{val:.4f}", va="center", fontsize=5.5)

plt.tight_layout()
os.makedirs("Resultados", exist_ok=True)
fig.savefig("Resultados/adaface_qmul_results_combined.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico salvo em Resultados/adaface_qmul_results_combined.png")


## 23. Análise Automática dos Resultados

In [ ]:
baseline_auc  = results.get("Baseline", {}).get("auc",   0.6992)
baseline_tar  = results.get("Baseline", {}).get("tar01", 0.2878)
baseline_t001 = results.get("Baseline", {}).get("tar001",0.0455)

acima_baseline = [(n, m) for n, m in results.items()
                  if m["auc"] > baseline_auc + 0.001]

print("=" * 60)
print("ANÁLISE DE RESULTADOS — GRUPOS A–I")
print("=" * 60)

print(f"\nBaseline: AUC={baseline_auc:.4f} | TAR@0.1={baseline_tar:.4f} | TAR@0.01={baseline_t001:.4f}")
print(f"\nTotal de experimentos: {len(results)}")

if acima_baseline:
    print(f"\nMétodos que SUPERARAM o baseline em AUC (+0.001):")
    for n, m in sorted(acima_baseline, key=lambda x: x[1]["auc"], reverse=True):
        print(f"  ▲ {n:35s}  AUC={m['auc']:.4f} (+{m['auc']-baseline_auc:.4f})")
else:
    print("\nNenhum método superou o baseline em AUC por margem > 0.001.")

# Melhores por grupo
print("\nMelhor por grupo (AUC):")
for grp in GROUP_ORDER:
    group_res = [(n, m) for n, m in results.items()
                 if exp_map.get(n, {}).get("group") == grp]
    if not group_res:
        continue
    best_n, best_m = max(group_res, key=lambda x: x[1]["auc"])
    delta = best_m["auc"] - baseline_auc
    sign  = "▲" if delta > 0.001 else ("▼" if delta < -0.001 else "=")
    print(f"  [{grp}] {best_n:35s}  AUC={best_m['auc']:.4f} {sign} ({delta:+.4f})")

# Melhor TAR@0.1
best_tar01 = max(results.items(), key=lambda x: x[1]["tar01"])
print(f"\nMelhor TAR@FAR=0.1: {best_tar01[0]} = {best_tar01[1]['tar01']:.4f}")

best_tar001 = max(results.items(), key=lambda x: x[1]["tar001"])
print(f"Melhor TAR@FAR=0.01: {best_tar001[0]} = {best_tar001[1]['tar001']:.4f}")
